# A3.4 — Gerekçe Şekilleri için Büyük LLM (Colab)

Bu notebook **Google Colab'da**, ücretsiz T4 GPU ile çalışacak şekilde yazıldı.

**Ne yapıyor:** `karar_noktalari.jsonl`'daki 50.050 karar noktasının **hepsini
değil**, yalnızca arkasındaki **birkaç benzersiz ŞEKLİ** (karar tipi +
tetiklenen kural kodları — `decide.py` sabit bir kural dizisi ürettiği için
bu sayı genelde 3'tür) büyük bir LLM'e (Qwen2.5-7B-Instruct) verip yer tutucu
(placeholder) token'lı Türkçe cümle varyantları üretiyor. Gerçek sayılar bu
varyantlara sonra **yerelde, GPU'suz** basılıyor (`training/label_rationale.py`).

> Bu, A3.3'teki (`paraphrase_colab.ipynb`) router-şablon başkalaştırma
> notebook'uyla aynı desen: 25 saat sürecek bir işi onlarca LLM çağrısına
> indirgemek.

**Bilgisayarında RAM/GPU yoksa bu notebook tam olarak bunun için var** —
yalnızca birkaç KB'lık bir prompt dosyası taşınıyor, 50.050 satırlık veri
seti hiç Colab'a gitmiyor.

## 1. Ortamı hazırla

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

## 2. Şekil prompt dosyasını yükle

Önce yerelde şunu çalıştır (proje kökünde):

```bash
uv run python -m training.label_rationale --sekil-ihrac-et data/egitim/sekil_promptlari.jsonl
```

Aşağıdaki hücre bir dosya seçme penceresi açacak — `sekil_promptlari.jsonl`
dosyasını seç (birkaç satırlık, küçük bir dosya).

In [ ]:
from google.colab import files

yuklenen = files.upload()
sekil_dosyasi = next(iter(yuklenen.keys()))
print("Yüklenen dosya:", sekil_dosyasi)

## 3. Modeli yükle (4-bit, T4'e sığması için)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ADI = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ADI,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model yüklendi.")

## 4. Üretim fonksiyonu

`label_rationale.py::_sekil_prompt_olustur` ile aynı prompt zaten yüklenen
dosyada hazır geliyor — burada yalnızca modele soruyoruz. Yer tutucu/yankı/
format kontrolü **kasıtlı olarak burada yok** — o sorumluluk yerelde
`_varyantlari_ayikla`'da kalıyor (ham yanıt olduğu gibi indirilip yerelde
ayıklanacak), böylece kontrol mantığı tek bir yerde (projede, versiyon
kontrolünde) yaşıyor.

In [ ]:
def yanit_uret(prompt: str, max_new_tokens: int = 500) -> str:
    mesajlar = [{"role": "user", "content": prompt}]
    girdi_metni = tokenizer.apply_chat_template(
        mesajlar, tokenize=False, add_generation_prompt=True
    )
    girdi = tokenizer(girdi_metni, return_tensors="pt").to(model.device)
    cikti = model.generate(
        **girdi,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    yeni_token_sayisi = girdi["input_ids"].shape[1]
    return tokenizer.decode(cikti[0][yeni_token_sayisi:], skip_special_tokens=True).strip()

## 5. Tüm şekilleri işle

Yalnızca birkaç şekil olduğu için (genelde 3) örnekleme adımına gerek yok,
hepsini tek seferde işleyip sonucu gözden geçiriyoruz.

In [ ]:
import json
import time

with open(sekil_dosyasi, encoding="utf-8") as f:
    sekiller = [json.loads(satir) for satir in f]

print("Toplam şekil sayısı:", len(sekiller))

t0 = time.time()
sonuclar = []
for kayit in sekiller:
    yanit = yanit_uret(kayit["prompt"])
    sonuclar.append({**kayit, "yanit": yanit})
t1 = time.time()

print(f"Süre: {t1 - t0:.1f} sn")
print()
print("--- Tüm sonuçlar ---")
for s in sonuclar:
    print("Şekil:", s["sekil_id"])
    print("Yanıt:")
    print(s["yanit"])
    print()

## 6. Kalite kontrolü (elle)

**Durup yukarıdaki yanıtları oku.** Her yer tutucu (ör. `{SIPARIS_MIKTARI}`)
hâlâ süslü parantezli, harfi harfine metinde duruyor mu? Model talimatı kendi
cevabıymış gibi geri döndürüyor mu ("yer tutucu", "harfi harfine" gibi
kelimeler geçiyorsa bu bir yankı — `label_rationale.py::_YANKI_IFADELERI`
bunu yerelde zaten eleyecek, ama az sayıda varyant kalırsa 5. hücreyi tekrar
çalıştırıp sayıyı artırman gerekebilir). Sorunluysa `_sekil_prompt_olustur`'u
(projede, yerelde) güncelleyip şekil dosyasını yeniden çıkar.

## 7. Sonucu kaydet ve indir

In [ ]:
from google.colab import files

CIKTI_DOSYASI = "sekil_varyantlari.jsonl"

with open(CIKTI_DOSYASI, "w", encoding="utf-8") as f:
    for s in sonuclar:
        f.write(
            json.dumps(
                {
                    "sekil_id": s["sekil_id"],
                    "slot_tokenlari": s["slot_tokenlari"],
                    "yanit": s["yanit"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

print("Yazıldı:", CIKTI_DOSYASI)
files.download(CIKTI_DOSYASI)

## Sonrası

İndirdiğin `sekil_varyantlari.jsonl` dosyasını projede
`data/egitim/sekil_varyantlari.jsonl` olarak kaydet. Sonra yerelde,
**Ollama'ya hiç gerek kalmadan**, tam veri seti şununla işlenir:

```bash
uv run python -m training.label_rationale --varyant-girdi data/egitim/sekil_varyantlari.jsonl
```

Bu, önceden üretilmiş varyantları kullanır; yalnızca (varsa) yeni/tanınmayan
şekiller için Ollama'ya düşer — o da yoksa güvenle şablona
(`app/llm/explain.py::sablon_gerekce`) düşülür, veri hiçbir zaman kirlenmez.